# Testes da camada de persistência — Sistema Acadêmico

Notebook de avaliação da camada de persistência construída com **ORMLite** sobre **SQLite**,
seguindo o Capítulo 12 (*Mapeamento de objetos para o modelo relacional*).

O roteiro percorre, nesta ordem:

1. geração do esquema a partir das anotações das entidades;
2. `CREATE` — inserção de objetos;
3. associação **1:1** (curso ↔ coordenador) e a restrição `UNIQUE` que a sustenta;
4. associação **1:N** (curso → estudantes, curso → disciplinas, professor → disciplinas);
5. associação **N:M** (estudante ↔ disciplina) resolvida pela tabela de associação;
6. atributos da **classe associativa** e `UPDATE`;
7. materialização: reabrir o banco e reconstruir os objetos;
8. `DELETE` e integridade referencial;
9. conferência no nível relacional, com SQL puro;
10. relatório final dos testes.

**Pré-requisito:** empacotar o projeto antes de abrir o notebook, para que os *jars* existam.

```bash
mvn -q package -DskipTests
```

## 0. Carga da camada de persistência

As diretivas `%jars` do kernel IJava colocam no *classpath* o jar do projeto e os jars do
ORMLite e do driver SQLite, copiados para `target/dependency` pelo `maven-dependency-plugin`.

In [1]:
%jars ../target/orm-academico-1.0.0.jar
%jars ../target/dependency/*.jar

In [2]:
// Silencia o log do ORMLite (por padrao ele imprime, em DEBUG, todo SQL emitido).
// Precisa vir antes do primeiro uso do framework.
System.setProperty("com.j256.simplelogging.level", "ERROR");

import model.*;
import persistence.*;

import com.j256.ormlite.dao.GenericRawResults;
import com.j256.ormlite.stmt.QueryBuilder;

import java.sql.SQLException;
import java.time.LocalDate;
import java.util.ArrayList;
import java.util.List;

// --- mini arcabouco de verificacao usado ao longo do notebook -------------
List<String> falhas = new ArrayList<>();
int[] totalDeTestes = {0};

void verificar(String descricao, boolean condicao) {
    totalDeTestes[0]++;
    if (condicao) {
        System.out.println("[ OK ]    " + descricao);
    } else {
        falhas.add(descricao);
        System.out.println("[FALHA]   " + descricao);
    }
}

System.out.println("Camada carregada. Java " + System.getProperty("java.version"));

Camada carregada. Java 21.0.3


## 1. Conexão e geração do esquema

A classe `Persistencia` é a fachada da camada: abre a conexão e expõe um repositório por
entidade. `recriarEsquema()` apaga e recria as tabelas, de modo que o notebook sempre parte
de um estado conhecido.

O DDL abaixo **não foi escrito à mão**: o ORMLite o deriva das anotações das entidades.
É nele que se lê o resultado do mapeamento — `generatedId` virou
`INTEGER PRIMARY KEY AUTOINCREMENT`, `canBeNull = false` virou `NOT NULL`,
`unique = true` virou `UNIQUE`.

In [3]:
Persistencia persistencia = new Persistencia("academico-notebook.sqlite");
persistencia.recriarEsquema();

for (String comando : persistencia.ddl()) {
    System.out.println(comando);
    System.out.println();
}

CREATE TABLE `professor` (`id` INTEGER PRIMARY KEY AUTOINCREMENT , `matricula` VARCHAR NOT NULL , `nome` VARCHAR NOT NULL , `email` VARCHAR , `titulacao` VARCHAR ,  UNIQUE (`matricula`)) 


CREATE TABLE `curso` (`id` INTEGER PRIMARY KEY AUTOINCREMENT , `nome` VARCHAR NOT NULL , `sigla` VARCHAR NOT NULL , `carga_horaria` INTEGER , `coordenador_id` INTEGER ,  UNIQUE (`nome`),  UNIQUE (`sigla`),  UNIQUE (`coordenador_id`)) 


CREATE TABLE `estudante` (`id` INTEGER PRIMARY KEY AUTOINCREMENT , `matricula` VARCHAR NOT NULL , `nome_completo` VARCHAR NOT NULL , `data_nascimento` VARCHAR , `curso_id` INTEGER NOT NULL ,  UNIQUE (`matricula`)) 


CREATE TABLE `disciplina` (`id` INTEGER PRIMARY KEY AUTOINCREMENT , `codigo` VARCHAR NOT NULL , `nome` VARCHAR NOT NULL , `carga_horaria` INTEGER , `curso_id` INTEGER NOT NULL , `professor_id` INTEGER ,  UNIQUE (`codigo`)) 


CREATE TABLE `inscricao` (`id` INTEGER PRIMARY KEY AUTOINCREMENT , `estudante_id` INTEGER NOT NULL , `disciplina_id` INTEGER NOT NULL , `semestre` VARCHAR NOT NULL , `nota` DOUBLE PRECISION , `frequencia` INTEGER , UNIQUE (`estudante_id`,`disciplina_id`,`semestre`) ) 


In [4]:
String ddlCompleto = String.join("\n", persistencia.ddl());

verificar("a tabela curso recebeu a chave estrangeira do coordenador",
          ddlCompleto.contains("`coordenador_id` INTEGER"));
verificar("a coluna coordenador_id e UNIQUE (e o que torna a associacao 1:1)",
          ddlCompleto.contains("UNIQUE (`coordenador_id`)"));
verificar("a tabela estudante recebeu a chave estrangeira curso_id (1:N)",
          ddlCompleto.contains("`curso_id` INTEGER NOT NULL"));
verificar("a tabela de associacao inscricao tem as duas chaves estrangeiras (N:M)",
          ddlCompleto.contains("`estudante_id` INTEGER NOT NULL")
          && ddlCompleto.contains("`disciplina_id` INTEGER NOT NULL"));
verificar("a associacao N:M tem chave logica UNIQUE composta",
          ddlCompleto.contains("UNIQUE (`estudante_id`,`disciplina_id`,`semestre`)"));

[ OK ]    a tabela curso recebeu a chave estrangeira do coordenador


[ OK ]    a coluna coordenador_id e UNIQUE (e o que torna a associacao 1:1)


[ OK ]    a tabela estudante recebeu a chave estrangeira curso_id (1:N)


[ OK ]    a tabela de associacao inscricao tem as duas chaves estrangeiras (N:M)


[ OK ]    a associacao N:M tem chave logica UNIQUE composta


## 2. CREATE — inserção de objetos

Cada `create()` emite um `INSERT`. O identificador é gerado pelo `AUTOINCREMENT` do SQLite e
devolvido ao objeto Java pelo próprio ORM: antes da inserção `id` vale 0, depois vale a chave
primária da linha.

In [5]:
Professor ana    = new Professor("P001", "Ana Ribeiro",    "ana@ufg.br",    Titulacao.DOUTOR);
System.out.println("antes do INSERT  -> id = " + ana.getId());

persistencia.professores().create(ana);
System.out.println("depois do INSERT -> id = " + ana.getId());

Professor bruno  = persistencia.professores().create(
        new Professor("P002", "Bruno Carvalho", "bruno@ufg.br", Titulacao.MESTRE));
Professor carmem = persistencia.professores().create(
        new Professor("P003", "Carmem Dias",    "carmem@ufg.br", Titulacao.DOUTOR));

verificar("o id foi gerado pelo banco e devolvido ao objeto", ana.getId() > 0);
verificar("tres professores persistidos", persistencia.professores().count() == 3);

antes do INSERT  -> id = 0


depois do INSERT -> id = 1


[ OK ]    o id foi gerado pelo banco e devolvido ao objeto


[ OK ]    tres professores persistidos


In [6]:
Curso si = persistencia.cursos().create(new Curso("Sistemas de Informacao", "SI", 3200));
Curso cc = persistencia.cursos().create(new Curso("Ciencia da Computacao", "CC", 3400));

Estudante carla = persistencia.estudantes().create(
        new Estudante("2026001", "Carla Nunes",  LocalDate.of(2004, 3, 12), si));
Estudante diego = persistencia.estudantes().create(
        new Estudante("2026002", "Diego Alves",  LocalDate.of(2003, 9, 30), si));
Estudante elisa = persistencia.estudantes().create(
        new Estudante("2026003", "Elisa Moraes", LocalDate.of(2005, 1, 20), cc));

Disciplina poo = persistencia.disciplinas().create(
        new Disciplina("INF0285", "Programacao Orientada a Objetos", 64, si, ana));
Disciplina bd  = persistencia.disciplinas().create(
        new Disciplina("INF0287", "Banco de Dados", 64, si, bruno));
Disciplina eda = persistencia.disciplinas().create(
        new Disciplina("INF0292", "Estruturas de Dados", 96, cc, carmem));

System.out.println(si);
System.out.println(carla);
System.out.println(poo);

verificar("dois cursos, tres estudantes e tres disciplinas persistidos",
          persistencia.cursos().count() == 2
          && persistencia.estudantes().count() == 3
          && persistencia.disciplinas().count() == 3);

Curso{id=1, sigla='SI', nome='Sistemas de Informacao', coordenador=nenhum}


Estudante{id=1, matricula='2026001', nome='Carla Nunes', nascimento=2004-03-12, curso=SI}


Disciplina{id=1, codigo='INF0285', nome='Programacao Orientada a Objetos', cargaHoraria=64, professor=Ana Ribeiro}


[ OK ]    dois cursos, tres estudantes e tres disciplinas persistidos


## 3. Associação 1:1 — curso e coordenador

No modelo relacional a associação 1:1 é obtida com uma chave estrangeira **mais** uma
restrição `UNIQUE`: sem o `UNIQUE`, a coluna `coordenador_id` representaria um 1:N. O teste
abaixo verifica a navegação nos dois sentidos e, em seguida, que o banco **recusa** dois
cursos com o mesmo coordenador.

In [7]:
persistencia.cursos().definirCoordenador(si, ana);

Curso siLido = persistencia.cursos().buscarPorSigla("SI");
System.out.println("curso  -> coordenador : " + siLido.getCoordenador().getNome());

Curso cursoDeAna = persistencia.cursos().coordenadoPor(ana);
System.out.println("professor -> curso    : " + cursoDeAna.getNome());

verificar("navegacao curso -> coordenador",  siLido.getCoordenador().getId() == ana.getId());
verificar("navegacao coordenador -> curso",  cursoDeAna.getId() == si.getId());
verificar("o curso CC ainda esta sem coordenador",
          persistencia.cursos().semCoordenador().size() == 1);

curso  -> coordenador : Ana Ribeiro


professor -> curso    : Sistemas de Informacao


[ OK ]    navegacao curso -> coordenador


[ OK ]    navegacao coordenador -> curso


[ OK ]    o curso CC ainda esta sem coordenador


In [8]:
// Teste negativo: a restricao UNIQUE deve impedir que Ana coordene tambem o curso CC.
boolean recusado = false;
try {
    persistencia.cursos().definirCoordenador(cc, ana);
} catch (SQLException e) {
    recusado = true;
    System.out.println("recusado pelo SGBD: " + e.getMessage().split("\n")[0]);
}
persistencia.cursos().refresh(cc);

verificar("o banco recusa o mesmo professor coordenando dois cursos (1:1 preservado)",
          recusado && cc.getCoordenador() == null);

recusado pelo SGBD: Unable to run update stmt on object Curso{id=2, sigla='CC', nome='Ciencia da Computacao', coordenador=Ana Ribeiro}: UPDATE `curso` SET `nome` = ?, `sigla` = ?, `carga_horaria` = ?, `coordenador_id` = ? WHERE `id` = ?


[ OK ]    o banco recusa o mesmo professor coordenando dois cursos (1:1 preservado)


## 4. Associações 1:N

A chave estrangeira fica sempre na tabela do lado *muitos*. A consulta correspondente é um
`SELECT ... WHERE <chave_estrangeira> = ?`, encapsulado nos métodos do repositório.

A navegação inversa — do lado *um* para a coleção — usa `ForeignCollection`: o ORM só emite o
`SELECT` quando a coleção é percorrida.

In [9]:
System.out.println("Estudantes do curso " + si.getSigla() + ":");
for (Estudante e : persistencia.estudantes().doCurso(si)) {
    System.out.println("  " + e.getNomeCompleto() + "  (curso_id = " + e.getCurso().getId() + ")");
}

System.out.println();
System.out.println("Disciplinas do curso " + si.getSigla() + ":");
for (Disciplina d : persistencia.disciplinas().doCurso(si)) {
    System.out.println("  " + d.getCodigo() + " - " + d.getNome());
}

System.out.println();
System.out.println("Disciplinas ministradas por " + carmem.getNome() + ":");
for (Disciplina d : persistencia.disciplinas().ministradasPor(carmem)) {
    System.out.println("  " + d.getCodigo() + " - " + d.getNome());
}

verificar("o curso SI tem 2 estudantes",   persistencia.estudantes().doCurso(si).size() == 2);
verificar("o curso CC tem 1 estudante",    persistencia.estudantes().doCurso(cc).size() == 1);
verificar("o curso SI oferece 2 disciplinas", persistencia.disciplinas().doCurso(si).size() == 2);
verificar("Carmem ministra 1 disciplina",  persistencia.disciplinas().ministradasPor(carmem).size() == 1);

Estudantes do curso SI:


  Carla Nunes  (curso_id = 1)


  Diego Alves  (curso_id = 1)


Disciplinas do curso SI:


  INF0285 - Programacao Orientada a Objetos


...

  INF0292 - Estruturas de Dados


[ OK ]    o curso SI tem 2 estudantes


[ OK ]    o curso CC tem 1 estudante


[ OK ]    o curso SI oferece 2 disciplinas


[ OK ]    Carmem ministra 1 disciplina


In [10]:
// Navegacao pelo lado "um" da associacao, com carga tardia (ForeignCollection).
Curso siRecarregado = persistencia.cursos().loadFromId(si.getId());

System.out.println("Colecao curso.getEstudantes():");
for (Estudante e : siRecarregado.getEstudantes()) {
    System.out.println("  " + e.getMatricula() + " - " + e.getNomeCompleto());
}

verificar("a ForeignCollection do curso traz os mesmos 2 estudantes",
          siRecarregado.getEstudantes().size() == 2);

Colecao curso.getEstudantes():


  2026001 - Carla Nunes


  2026002 - Diego Alves


[ OK ]    a ForeignCollection do curso traz os mesmos 2 estudantes


## 5. Associação N:M — estudante e disciplina

A associação muitos-para-muitos não existe diretamente no modelo relacional: ela é
representada pela tabela de associação `inscricao`, com uma chave estrangeira para cada lado.
A camada de persistência esconde esse detalhe e devolve objetos de domínio.

In [11]:
String semestre = "2026/1";

Inscricao i1 = persistencia.inscricoes().matricular(carla, poo, semestre);
Inscricao i2 = persistencia.inscricoes().matricular(carla, bd,  semestre);
Inscricao i3 = persistencia.inscricoes().matricular(diego, poo, semestre);
Inscricao i4 = persistencia.inscricoes().matricular(elisa, eda, semestre);

System.out.println("Disciplinas cursadas por " + carla.getNomeCompleto() + ":");
for (Disciplina d : persistencia.inscricoes().disciplinasDe(carla)) {
    System.out.println("  " + d.getCodigo() + " - " + d.getNome());
}

System.out.println();
System.out.println("Estudantes inscritos em " + poo.getCodigo() + ":");
for (Estudante e : persistencia.inscricoes().estudantesDe(poo)) {
    System.out.println("  " + e.getMatricula() + " - " + e.getNomeCompleto());
}

verificar("Carla cursa 2 disciplinas", persistencia.inscricoes().disciplinasDe(carla).size() == 2);
verificar("POO tem 2 estudantes",      persistencia.inscricoes().estudantesDe(poo).size() == 2);
verificar("quatro linhas na tabela de associacao", persistencia.inscricoes().count() == 4);

Disciplinas cursadas por Carla Nunes:


  INF0285 - Programacao Orientada a Objetos


  INF0287 - Banco de Dados


Estudantes inscritos em INF0285:


  2026001 - Carla Nunes


  2026002 - Diego Alves


[ OK ]    Carla cursa 2 disciplinas


[ OK ]    POO tem 2 estudantes


[ OK ]    quatro linhas na tabela de associacao


In [12]:
// O SQL que o ORM monta para navegar o N:M: uma juncao com a tabela de associacao.
QueryBuilder<Inscricao, Integer> qbInscricao = persistencia.inscricoes().queryBuilder();
qbInscricao.where().eq("estudante_id", carla.getId());

QueryBuilder<Disciplina, Integer> qbDisciplina = persistencia.disciplinas().queryBuilder();
String sqlDaJuncao = qbDisciplina.join(qbInscricao).prepare().getStatement();

System.out.println(sqlDaJuncao);

verificar("a navegacao N:M e resolvida por INNER JOIN na tabela de associacao",
          sqlDaJuncao.contains("INNER JOIN") && sqlDaJuncao.contains("inscricao"));

SELECT `disciplina`.* FROM `disciplina` INNER JOIN `inscricao` ON `disciplina`.`id` = `inscricao`.`disciplina_id` WHERE `inscricao`.`estudante_id` = 1


[ OK ]    a navegacao N:M e resolvida por INNER JOIN na tabela de associacao


In [13]:
// Teste negativo: a chave logica UNIQUE (estudante, disciplina, semestre) impede
// que o mesmo estudante seja inscrito duas vezes na mesma disciplina no mesmo semestre.
boolean duplicataRecusada = false;
try {
    persistencia.inscricoes().matricular(carla, poo, semestre);
} catch (SQLException e) {
    duplicataRecusada = true;
    System.out.println("recusado pelo SGBD: " + e.getMessage().split("\n")[0]);
}

verificar("inscricao duplicada e recusada", duplicataRecusada);
verificar("a tabela de associacao continua com 4 linhas",
          persistencia.inscricoes().count() == 4);

// A mesma dupla em OUTRO semestre e legitima (repeticao da disciplina).
Inscricao repeticao = persistencia.inscricoes().matricular(carla, poo, "2026/2");
verificar("a mesma dupla em outro semestre e aceita", repeticao.getId() > 0);

// A juncao produz uma linha por inscricao; a lista de disciplinas cursadas,
// porem, nao pode repetir a disciplina feita em dois semestres (DISTINCT).
verificar("a navegacao N:M nao repete a disciplina cursada duas vezes",
          persistencia.inscricoes().disciplinasDe(carla).size() == 2);
verificar("mas as duas inscricoes continuam registradas",
          persistencia.inscricoes().historicoDe(carla).size() == 3);

recusado pelo SGBD: Unable to run insert stmt on object Inscricao{id=0, estudante=2026001, disciplina=INF0285, semestre='2026/1', nota=null, frequencia=0}: INSERT INTO `inscricao` (`estudante_id` ,`disciplina_id` ,`semestre` ,`nota` ,`frequencia` ) VALUES (?,?,?,?,?)


[ OK ]    inscricao duplicada e recusada


[ OK ]    a tabela de associacao continua com 4 linhas


[ OK ]    a mesma dupla em outro semestre e aceita


[ OK ]    a navegacao N:M nao repete a disciplina cursada duas vezes


[ OK ]    mas as duas inscricoes continuam registradas


## 6. Classe associativa e `UPDATE`

`semestre`, `nota` e `frequencia` são atributos da associação, não do estudante nem da
disciplina: só fazem sentido para o par. No mapeamento eles viraram colunas da tabela de
associação.

In [14]:
persistencia.inscricoes().lancarResultado(i1, 8.5, 92);   // Carla em POO
persistencia.inscricoes().lancarResultado(i2, 5.0, 80);   // Carla em BD
persistencia.inscricoes().lancarResultado(i3, 9.0, 88);   // Diego em POO

System.out.println("Historico de " + carla.getNomeCompleto() + ":");
for (Inscricao ins : persistencia.inscricoes().historicoDe(carla)) {
    System.out.println(String.format("  %-8s %-8s nota=%-5s freq=%3d%%  %s",
            ins.getSemestre(),
            ins.getDisciplina().getCodigo(),
            ins.getNota() == null ? "-" : ins.getNota(),
            ins.getFrequencia(),
            ins.isAprovado() ? "aprovado" : "nao aprovado"));
}

List<Inscricao> aprovadosPoo = persistencia.inscricoes().aprovadosEm(poo, semestre);
System.out.println();
System.out.println("Aprovados em " + poo.getCodigo() + " no semestre " + semestre + ": "
        + aprovadosPoo.size());

verificar("o UPDATE gravou a nota na tabela de associacao",
          persistencia.inscricoes().loadFromId(i1.getId()).getNota() == 8.5);
verificar("historico de Carla com 3 inscricoes",
          persistencia.inscricoes().historicoDe(carla).size() == 3);
verificar("2 aprovados em POO", aprovadosPoo.size() == 2);
verificar("Carla reprovada em BD por nota", !persistencia.inscricoes().loadFromId(i2.getId()).isAprovado());

Historico de Carla Nunes:


  2026/1   INF0285  nota=8.5   freq= 92%  aprovado


  2026/1   INF0287  nota=5.0   freq= 80%  nao aprovado


  2026/2   INF0285  nota=-     freq=  0%  nao aprovado


Aprovados em INF0285 no semestre 2026/1: 2


[ OK ]    o UPDATE gravou a nota na tabela de associacao


[ OK ]    historico de Carla com 3 inscricoes


[ OK ]    2 aprovados em POO


[ OK ]    Carla reprovada em BD por nota


## 7. Materialização — fechar e reabrir o banco

Este é o teste que separa objeto **transiente** de objeto **persistente**: a conexão é
encerrada, todas as referências em memória são descartadas e os objetos são reconstruídos a
partir das linhas gravadas, com as associações intactas.

In [15]:
persistencia.close();
System.out.println("conexao encerrada");

Persistencia novaSessao = new Persistencia("academico-notebook.sqlite");

Estudante carlaMaterializada = novaSessao.estudantes().buscarPorMatricula("2026001");
System.out.println("objeto reconstruido: " + carlaMaterializada);
System.out.println("curso ..............: " + carlaMaterializada.getCurso().getNome());
System.out.println("coordenador do curso: " + carlaMaterializada.getCurso().getCoordenador().getNome());

System.out.println();
System.out.println("disciplinas (N:M) ..:");
for (Disciplina d : novaSessao.inscricoes().disciplinasDe(carlaMaterializada)) {
    System.out.println("  " + d.getCodigo() + " ministrada por " + d.getProfessor().getNome());
}

verificar("o estudante foi materializado a partir do banco",
          carlaMaterializada != null && carlaMaterializada.getId() == carla.getId());
verificar("a associacao 1:N sobreviveu ao fechamento da conexao",
          "SI".equals(carlaMaterializada.getCurso().getSigla()));
verificar("a associacao 1:1 sobreviveu (curso -> coordenador)",
          "Ana Ribeiro".equals(carlaMaterializada.getCurso().getCoordenador().getNome()));
verificar("a associacao N:M sobreviveu",
          novaSessao.inscricoes().disciplinasDe(carlaMaterializada).size() == 2);
verificar("a data de nascimento voltou com o mesmo valor",
          LocalDate.of(2004, 3, 12).equals(carlaMaterializada.getDataNascimentoComoLocalDate()));

conexao encerrada


objeto reconstruido: Estudante{id=1, matricula='2026001', nome='Carla Nunes', nascimento=2004-03-12, curso=SI}


curso ..............: Sistemas de Informacao


coordenador do curso: Ana Ribeiro


disciplinas (N:M) ..:


...

[ OK ]    o estudante foi materializado a partir do banco


[ OK ]    a associacao 1:N sobreviveu ao fechamento da conexao


[ OK ]    a associacao 1:1 sobreviveu (curso -> coordenador)


[ OK ]    a associacao N:M sobreviveu


[ OK ]    a data de nascimento voltou com o mesmo valor


## 8. DELETE e integridade referencial

Remover uma inscrição desfaz apenas a associação: os dois objetos associados continuam
existindo.

A remoção de um objeto **referenciado** por outras linhas é mais delicada. O DDL que o
ORMLite gera para o SQLite cria as colunas de chave estrangeira, mas não declara a restrição
`FOREIGN KEY`: o banco aceitaria o `DELETE` e deixaria linhas órfãs. Por isso a camada de
persistência assume essa responsabilidade, de duas formas:

- `CursoRepositorio.delete()` **recusa** a remoção de um curso ainda referenciado;
- `Persistencia.removerEstudanteEmCascata()` apaga primeiro as linhas da associação e só
  então o estudante — a remoção **em cascata** discutida no mapeamento de agregações.

In [16]:
// (a) Cancelar uma inscricao: desfaz apenas a associacao.
long antes = novaSessao.inscricoes().count();

Estudante carlaLida = novaSessao.estudantes().buscarPorMatricula("2026001");
Inscricao repeticaoLida = novaSessao.inscricoes().historicoDe(carlaLida).stream()
        .filter(ins -> "2026/2".equals(ins.getSemestre()))
        .findFirst().orElseThrow();

novaSessao.inscricoes().cancelar(repeticaoLida);
long depois = novaSessao.inscricoes().count();
System.out.println("inscricoes antes = " + antes + ", depois do cancelamento = " + depois);

verificar("o DELETE removeu uma linha da tabela de associacao", depois == antes - 1);
verificar("cancelar a inscricao nao apagou o estudante",
          novaSessao.estudantes().buscarPorMatricula("2026001") != null);
verificar("cancelar a inscricao nao apagou a disciplina",
          novaSessao.disciplinas().buscarPorCodigo("INF0285") != null);

inscricoes antes = 5, depois do cancelamento = 4


[ OK ]    o DELETE removeu uma linha da tabela de associacao


[ OK ]    cancelar a inscricao nao apagou o estudante


[ OK ]    cancelar a inscricao nao apagou a disciplina


In [17]:
// (b) Remover um curso ainda referenciado deve ser recusado pela camada.
Curso ccLido = novaSessao.cursos().buscarPorSigla("CC");
System.out.println("o curso CC e referenciado por "
        + novaSessao.cursos().dependentesDe(ccLido) + " registro(s)");

boolean remocaoBloqueada = false;
try {
    novaSessao.cursos().delete(ccLido);
} catch (SQLException e) {
    remocaoBloqueada = true;
    System.out.println("recusado pela camada: " + e.getMessage());
}

verificar("a remocao de um curso com dependentes e recusada", remocaoBloqueada);
verificar("o curso continua no banco", novaSessao.cursos().buscarPorSigla("CC") != null);

o curso CC e referenciado por 2 registro(s)


recusado pela camada: O curso CC nao pode ser removido: 2 registro(s) ainda o referenciam


[ OK ]    a remocao de um curso com dependentes e recusada


[ OK ]    o curso continua no banco


In [18]:
// (c) Remocao em cascata: as inscricoes do estudante saem antes dele.
Estudante elisaLida = novaSessao.estudantes().buscarPorMatricula("2026003");
long inscricoesAntes = novaSessao.inscricoes().count();

int inscricoesRemovidas = novaSessao.removerEstudanteEmCascata(elisaLida);
System.out.println("inscricoes removidas em cascata: " + inscricoesRemovidas);

verificar("a cascata removeu a inscricao do estudante", inscricoesRemovidas == 1);
verificar("o estudante foi removido",
          novaSessao.estudantes().buscarPorMatricula("2026003") == null);
verificar("a tabela de associacao perdeu exatamente uma linha",
          novaSessao.inscricoes().count() == inscricoesAntes - 1);

inscricoes removidas em cascata: 1


[ OK ]    a cascata removeu a inscricao do estudante


[ OK ]    o estudante foi removido


[ OK ]    a tabela de associacao perdeu exatamente uma linha


In [19]:
// (d) Sem dependentes, a remocao do curso passa a ser aceita.
Disciplina edaLida = novaSessao.disciplinas().buscarPorCodigo("INF0292");
novaSessao.disciplinas().delete(edaLida);

Curso ccSemDependentes = novaSessao.cursos().buscarPorSigla("CC");
System.out.println("dependentes do curso CC agora: "
        + novaSessao.cursos().dependentesDe(ccSemDependentes));

long cursosAntes = novaSessao.cursos().count();
novaSessao.cursos().delete(ccSemDependentes);

verificar("sem dependentes, o curso e removido",
          novaSessao.cursos().count() == cursosAntes - 1);
verificar("o id removido nao e mais encontrado",
          novaSessao.cursos().loadFromId(ccSemDependentes.getId()) == null);

dependentes do curso CC agora: 0


[ OK ]    sem dependentes, o curso e removido


[ OK ]    o id removido nao e mais encontrado


## 9. Conferência no nível relacional

Até aqui tudo foi observado pela camada de persistência. Esta seção olha o banco por baixo
dela, com SQL puro, para confirmar que as linhas e as chaves estrangeiras são mesmo o que os
diagramas descrevem.

In [20]:
String consulta =
    "SELECT e.matricula, e.nome_completo, d.codigo, i.semestre, i.nota " +
    "FROM inscricao i " +
    "JOIN estudante  e ON e.id = i.estudante_id " +
    "JOIN disciplina d ON d.id = i.disciplina_id " +
    "ORDER BY e.matricula, i.semestre, d.codigo";

System.out.println(String.format("%-10s %-14s %-11s %-9s %s",
        "MATRICULA", "ESTUDANTE", "DISCIPLINA", "SEMESTRE", "NOTA"));
int linhas = 0;
try (GenericRawResults<String[]> resultado =
        novaSessao.inscricoes().getDao().queryRaw(consulta)) {
    for (String[] coluna : resultado) {
        linhas++;
        System.out.println(String.format("%-10s %-14s %-11s %-9s %s",
                coluna[0], coluna[1], coluna[2], coluna[3], coluna[4] == null ? "-" : coluna[4]));
    }
}

verificar("o JOIN em SQL puro devolve as mesmas inscricoes que o ORM",
          linhas == novaSessao.inscricoes().count() && linhas == 3);

MATRICULA  ESTUDANTE      DISCIPLINA  SEMESTRE  NOTA


2026001    Carla Nunes    INF0285     2026/1    8.5


2026001    Carla Nunes    INF0287     2026/1    5.0


2026002    Diego Alves    INF0285     2026/1    9.0


[ OK ]    o JOIN em SQL puro devolve as mesmas inscricoes que o ORM


In [21]:
// As colunas fisicas da tabela de associacao, sem intermediacao do ORM.
System.out.println("PRAGMA table_info(inscricao):");
try (GenericRawResults<String[]> info =
        novaSessao.inscricoes().getDao().queryRaw("PRAGMA table_info(inscricao)")) {
    for (String[] coluna : info) {
        System.out.println(String.format("  %-14s %-18s %s", coluna[1], coluna[2],
                "1".equals(coluna[3]) ? "NOT NULL" : ""));
    }
}

PRAGMA table_info(inscricao):


  id             INTEGER            


  estudante_id   INTEGER            NOT NULL


  disciplina_id  INTEGER            NOT NULL


  semestre       VARCHAR            NOT NULL


  nota           DOUBLE PRECISION   


  frequencia     INTEGER            


## 10. Relatório final

In [22]:
System.out.println("Linhas por tabela ao final do roteiro:");
System.out.println("  professor  = " + novaSessao.professores().count());
System.out.println("  curso      = " + novaSessao.cursos().count());
System.out.println("  estudante  = " + novaSessao.estudantes().count());
System.out.println("  disciplina = " + novaSessao.disciplinas().count());
System.out.println("  inscricao  = " + novaSessao.inscricoes().count());
System.out.println();

System.out.println("Testes executados : " + totalDeTestes[0]);
System.out.println("Falhas            : " + falhas.size());
for (String falha : falhas) {
    System.out.println("  - " + falha);
}
System.out.println();
System.out.println(falhas.isEmpty()
        ? "RESULTADO: camada de persistencia aprovada em todos os testes."
        : "RESULTADO: ha falhas a corrigir.");

novaSessao.close();

Linhas por tabela ao final do roteiro:


  professor  = 3


  curso      = 1


  estudante  = 2


  disciplina = 2


  inscricao  = 3


Testes executados : 46


Falhas            : 0


RESULTADO: camada de persistencia aprovada em todos os testes.
